# 1. **SETUP**

In [ ]:
import sklearn; print(sklearn.__version__)


In [ ]:
# Colab: install deps (scikit-learn + joblib)
!pip -q install scikit-learn joblib pandas numpy


# 2. Upload your **CSVs**

In [ ]:
import pandas as pd

genshin = pd.read_csv(
	"C:\\Users\\Elite BooK\\OneDrive\\Desktop\\ProjectForSummer\\genshin.csv",
	encoding='latin-1',
	low_memory=False
)
wuthering = pd.read_csv(
	"C:\\Users\\Elite BooK\\OneDrive\\Desktop\\ProjectForSummer\\wutheringwaves_character.csv",
	encoding='latin-1',
	low_memory=False
)

print(genshin.head())

# 3. **Load** & **Harmonize**

In [ ]:
import pandas as pd
import numpy as np

# Load
genshin = pd.read_csv('genshin.csv', encoding='latin-1', low_memory=False)
wuwa    = pd.read_csv('wutheringwaves_character.csv', encoding='latin-1', low_memory=False)

# --- Map to a unified schema ---
# Target to numeric 4/5
genshin_target = genshin['rarity'].astype(int)
wuwa_target = (wuwa['Rarity']
               .astype(str)
               .str.extract(r'(\d)')
               .astype(float)
               .astype('Int64')
               .fillna(0)
               .astype(int))  # "5 Star" -> 5, "4 Star" -> 4

# Features we will harmonize:
# game, weapon, element, role, region, affiliation, model_size, hp, atk, def
g = pd.DataFrame({
    'game':        'Genshin',
    'weapon':      genshin.get('weapon_type'),
    'element':     genshin.get('vision'),
    'role':        np.nan,  # not in genshin CSV
    'region':      genshin.get('region'),
    'affiliation': np.nan,  # not in genshin CSV (unless your CSV has it)
    'model_size':  genshin.get('model'),
    'hp':          genshin.get('hp_1_20'),
    'atk':         genshin.get('atk_1_20'),
    'def':         genshin.get('def_1_20'),
    'character':   genshin.get('character_name', pd.Series([np.nan]*len(genshin)))
})
g['rarity'] = genshin_target

w = pd.DataFrame({
    'game':        'Wuthering',
    'weapon':      wuwa.get('Weapon'),
    'element':     wuwa.get('Attribute'),
    'role':        wuwa.get('Role'),
    'region':      wuwa.get('Birthplace'),
    'affiliation': wuwa.get('Affiliation'),
    'model_size':  np.nan,  # not present in wuwa CSV
    'hp':          wuwa.get('HP'),
    'atk':         wuwa.get('ATK'),
    'def':         wuwa.get('DEF'),
    'character':   wuwa.get('Character')
})
w['rarity'] = wuwa_target

# Keep only 4★/5★ rows
g = g[g['rarity'].isin([4,5])].copy()
w = w[w['rarity'].isin([4,5])].copy()

combined = pd.concat([g, w], ignore_index=True)

print(combined.head())
print(combined.isna().mean().sort_values(ascending=False).head(10))
print("Shape:", combined.shape, "| Genshin rows:", g.shape[0], "| WuWa rows:", w.shape[0])


# 4. **Train**/**Test** **Split**, **Pipeline**, **Train**

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

X = combined.drop(columns=['rarity'])
y = combined['rarity'].astype(int)

cat_cols = X.select_dtypes(include=['object']).columns.tolist()
num_cols = X.select_dtypes(exclude=['object']).columns.tolist()

preprocess = ColumnTransformer(
    transformers=[
        ('num', SimpleImputer(strategy='median'), num_cols),
        ('cat', Pipeline([
            ('imp', SimpleImputer(strategy='most_frequent')),
            ('oh', OneHotEncoder(handle_unknown='ignore'))
        ]), cat_cols),
    ]
)

model = RandomForestClassifier(
    n_estimators=500,
    random_state=42,
    class_weight='balanced_subsample'
)

pipe = Pipeline([('prep', preprocess), ('clf', model)])

# Stratify by rarity (and roughly by game via group split trick)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

pipe.fit(X_train, y_train)
pred = pipe.predict(X_test)

print("Accuracy:", round(accuracy_score(y_test, pred), 4))
print("\nClassification Report:\n", classification_report(y_test, pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, pred))


# 5. **Per**-**Game** **Evaluation**

In [ ]:
X_test_eval = X_test.copy()
X_test_eval['true'] = y_test.values
X_test_eval['pred'] = pred

for game in X_test_eval['game'].dropna().unique():
    mask = X_test_eval['game'] == game
    yt = X_test_eval.loc[mask, 'true']
    yp = X_test_eval.loc[mask, 'pred']
    if len(yt) > 0:
        print(f"\n=== {game} ===")
        print("n =", len(yt), "| Acc:", round(accuracy_score(yt, yp), 4))
        print(classification_report(yt, yp))


# 6. **Save the Model**

In [ ]:
import joblib
joblib.dump(pipe, 'combined_model.pkl')
print("Saved combined model → combined_model.pkl")


# 7. **Download the model to your PC**

In [ ]:
from google.colab import files
files.download('combined_model.pkl')
